# Burst T+0 Simulator

Simula entrar en el **mismo día del burst** cuando el ticker aparece por primera vez
en Finviz Top Gainers, y compara:

- **Grupo A (filtrado)**: bursts que pasan `no_NewHigh_7d AND has_edgar_7d`
- **Grupo B (todo)**: los 177 bursts sin filtro
- **Grupo C (excluidos)**: bursts que NO pasan el filtro

Parámetros configurables:
- `STOP_PCT`: stop loss desde entrada (default -8%)
- `TARGET_PCT`: target opcional (default None = solo EOD)
- `MAX_ENTRY_MSO`: máximo minutos desde apertura para entrar (default 60)
- `CAPITAL`: capital por trade en $ (default 1000)


## 1. Configuración

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# ── Parámetros del simulador ──────────────────────────────────────────────────
STOP_PCT       = -0.08    # -8% stop loss desde entrada
TARGET_PCT     = None     # None = solo EOD exit; 0.20 = target +20%
MAX_ENTRY_MSO  = 60       # solo entrar si primera aparicion <= 60 min desde apertura
CAPITAL        = 1_000    # $ por trade

# ── Paths ──────────────────────────────────────────────────────────────────────
ANALYSIS_DIR   = Path('.').resolve().parent
FINVIZ_DB      = Path.home() / 'Library/Application Support/finviz-dashboard/finviz_snapshots.db'
INTRADAY_DB    = ANALYSIS_DIR / 'burst_intraday_cache.db'
DAILY_DB       = ANALYSIS_DIR / 'burst_daily_cache.db'

print(f"INTRADAY_DB exists: {INTRADAY_DB.exists()}")
print(f"DAILY_DB    exists: {DAILY_DB.exists()}")


## 2. Cargar datos

In [ ]:
import sys
sys.path.insert(0, str(Path('.').resolve()))

# ── 2a. Snapshots Finviz: primera aparición por (ticker, date) ─────────────────
with sqlite3.connect(FINVIZ_DB) as conn:
    snaps = pd.read_sql("""
        SELECT
            ticker,
            SUBSTR(timestamp,1,10)                                            AS date,
            MIN(timestamp)                                                    AS first_ts,
            CAST(REPLACE(REPLACE(change_pct,'%',''),',','') AS REAL)          AS change_pct,
            CAST(REPLACE(price,',','') AS REAL)                               AS price
        FROM snapshots
        WHERE category = 'Top Gainers'
          AND CAST(REPLACE(REPLACE(change_pct,'%',''),',','') AS REAL) >= 15
        GROUP BY ticker, SUBSTR(timestamp,1,10)
    """, conn)

TZ = 'America/New_York'
snaps['first_ts_et'] = pd.to_datetime(snaps['first_ts'], utc=True).dt.tz_convert(TZ)
snaps['hour']        = snaps['first_ts_et'].dt.hour
snaps['minute']      = snaps['first_ts_et'].dt.minute
snaps['mso']         = (snaps['hour'] - 9) * 60 + snaps['minute'] - 30
snaps = snaps[snaps['mso'] >= 0].copy()  # excluir pre-market

print(f"Burst events con primera aparicion: {len(snaps)}")
print(f"MSO distribution: min={snaps['mso'].min()}, median={snaps['mso'].median():.0f}, max={snaps['mso'].max()}")
print(f"Dentro de MAX_ENTRY_MSO={MAX_ENTRY_MSO}: {(snaps['mso'] <= MAX_ENTRY_MSO).sum()}")


In [ ]:
# ── 2b. Intraday bars 1-min ────────────────────────────────────────────────────
with sqlite3.connect(INTRADAY_DB) as conn:
    bars = pd.read_sql(
        "SELECT symbol AS ticker, date, dt, open, high, low, close, volume FROM bars",
        conn
    )

bars['bar_ts'] = pd.to_datetime(bars['date'] + 'T' + bars['dt']).dt.tz_localize(TZ, ambiguous='NaT', nonexistent='NaT')
bars['mso_bar'] = (bars['bar_ts'].dt.hour - 9) * 60 + bars['bar_ts'].dt.minute - 30
bars = bars[bars['mso_bar'] >= 0].copy()

intraday_pairs = set(zip(bars['ticker'], bars['date']))
print(f"Intraday bars: {len(bars):,} barras, {len(intraday_pairs)} ticker-days")


In [ ]:
# ── 2c. Daily bars: prev_close para calcular gap ───────────────────────────────
with sqlite3.connect(DAILY_DB) as conn:
    daily = pd.read_sql(
        "SELECT ticker, bar_date, open, high, low, close FROM daily_bars ORDER BY ticker, bar_date",
        conn
    )

daily['prev_close'] = daily.groupby('ticker')['close'].shift(1)
daily_idx = daily.set_index(['ticker', 'bar_date'])
print(f"Daily bars: {len(daily):,} barras, {daily['ticker'].nunique()} tickers")


## 3. Cargar filtros precursores

In [ ]:
# Cargar narrative_master.csv (construido en burst_narrative_reconstruction.ipynb)
# Contiene: ticker, burst_date, setup_type, nh_flag, n_filings_90d, burst_change_pct
NARRATIVE_CSV = Path('.').resolve() / 'narrative_master.csv'

if NARRATIVE_CSV.exists():
    narrative = pd.read_csv(NARRATIVE_CSV)
    print(f"narrative_master.csv: {len(narrative)} filas")
    print("Columnas:", list(narrative.columns))
else:
    print("narrative_master.csv no encontrado — generando desde Finviz DB...")
    # Fallback: usar snaps sin filtros precursores
    narrative = snaps[['ticker','date']].rename(columns={'date':'burst_date'}).copy()
    narrative['nh_flag']       = False
    narrative['n_filings_90d'] = 0
    narrative['setup_type']    = 'unknown'
    narrative['burst_change_pct'] = snaps['change_pct'].values


In [ ]:
# Merge: snaps (tiene MSO) + narrative (tiene filtros)
events = snaps.rename(columns={'date': 'burst_date'}).merge(
    narrative[['ticker','burst_date','setup_type','nh_flag','n_filings_90d','burst_change_pct']].drop_duplicates(),
    on=['ticker','burst_date'],
    how='left'
)

# Filtros precursores
events['no_nh']      = ~events['nh_flag'].fillna(False)
events['has_edgar']  = events['n_filings_90d'].fillna(0) > 0
events['filter_pass'] = events['no_nh'] & events['has_edgar']

# Solo bursts con datos intraday disponibles
events['has_intraday'] = events.apply(lambda r: (r['ticker'], r['burst_date']) in intraday_pairs, axis=1)

print(f"Total burst events:              {len(events)}")
print(f"  Con datos intraday:            {events['has_intraday'].sum()}")
print(f"  Con MSO <= {MAX_ENTRY_MSO}:              {(events['mso'] <= MAX_ENTRY_MSO).sum()}")
print(f"  Pasan filtro (no_nh+edgar):    {events['filter_pass'].sum()}")
print(f"  Pasan filtro + intraday:       {(events['filter_pass'] & events['has_intraday']).sum()}")
print(f"  No pasan filtro + intraday:    {(~events['filter_pass'] & events['has_intraday']).sum()}")


## 4. Motor del simulador

In [ ]:
def simulate_trade(ticker, burst_date, entry_mso, bars_df,
                   stop_pct=STOP_PCT, target_pct=TARGET_PCT,
                   trail_pct=None, capital=CAPITAL):
    """
    Simula un trade T+0 con soporte para trailing stop.

    trail_pct: si se especifica (ej. 0.08), usa trailing stop en lugar de stop fijo.
               El stop se mueve al alza siguiendo el precio máximo alcanzado.
    stop_pct : stop fijo inicial (también actúa como stop inicial del trailing).
    """
    day_bars = bars_df[
        (bars_df['ticker'] == ticker) & (bars_df['date'] == burst_date)
    ].sort_values('mso_bar').reset_index(drop=True)

    if day_bars.empty:
        return None

    entry_bars = day_bars[day_bars['mso_bar'] >= entry_mso]
    if entry_bars.empty:
        return None

    entry_bar        = entry_bars.iloc[0]
    entry_price      = entry_bar['close']
    entry_mso_actual = int(entry_bar['mso_bar'])

    if entry_price <= 0:
        return None

    # Stop inicial
    active_stop  = entry_price * (1 + stop_pct)
    target_price = entry_price * (1 + target_pct) if target_pct else None
    shares       = max(1, int(capital / entry_price))
    peak_price   = entry_price

    subsequent  = day_bars[day_bars['mso_bar'] > entry_mso_actual].reset_index(drop=True)
    exit_price  = None
    exit_reason = 'EOD'
    exit_mso    = entry_mso_actual
    mfe         = entry_price
    mae         = entry_price

    for _, bar in subsequent.iterrows():
        mfe = max(mfe, bar['high'])
        mae = min(mae, bar['low'])

        # Actualizar trailing stop
        if trail_pct and bar['high'] > peak_price:
            peak_price  = bar['high']
            trail_stop  = peak_price * (1 - trail_pct)
            active_stop = max(active_stop, trail_stop)  # solo sube, nunca baja

        # Stop hit
        if bar['low'] <= active_stop:
            exit_price  = active_stop
            exit_reason = 'TRAIL_STOP' if trail_pct else 'STOP'
            exit_mso    = int(bar['mso_bar'])
            break

        # Target hit
        if target_price and bar['high'] >= target_price:
            exit_price  = target_price
            exit_reason = 'TARGET'
            exit_mso    = int(bar['mso_bar'])
            break

    if exit_price is None:
        last_bar    = subsequent.iloc[-1] if not subsequent.empty else entry_bar
        exit_price  = last_bar['close']
        exit_reason = 'EOD'
        exit_mso    = int(last_bar['mso_bar'])

    pnl_pct = (exit_price - entry_price) / entry_price
    pnl_usd = (exit_price - entry_price) * shares

    return {
        'ticker':       ticker,
        'burst_date':   burst_date,
        'entry_price':  round(entry_price, 4),
        'exit_price':   round(exit_price, 4),
        'exit_reason':  exit_reason,
        'entry_mso':    entry_mso_actual,
        'exit_mso':     exit_mso,
        'pnl_pct':      round(pnl_pct * 100, 2),
        'pnl_usd':      round(pnl_usd, 2),
        'mfe_pct':      round((mfe - entry_price) / entry_price * 100, 2),
        'mae_pct':      round((mae - entry_price) / entry_price * 100, 2),
        'shares':       shares,
        'peak_stop':    round(active_stop, 4),
    }

print("simulate_trade (con trailing stop) definida.")


## 5. Ejecutar simulación

In [ ]:
# Filtrar eventos simulables
sim_events = events[
    events['has_intraday'] & (events['mso'] <= MAX_ENTRY_MSO)
].copy()

print(f"Eventos simulables: {len(sim_events)}")
print(f"  Pasan filtro:     {sim_events['filter_pass'].sum()}")
print(f"  No pasan filtro:  {(~sim_events['filter_pass']).sum()}")
print()

results = []
for _, ev in sim_events.iterrows():
    r = simulate_trade(
        ticker     = ev['ticker'],
        burst_date = ev['burst_date'],
        entry_mso  = int(ev['mso']),
        bars_df    = bars,
    )
    if r is None:
        continue
    r['filter_pass'] = ev['filter_pass']
    r['setup_type']  = ev.get('setup_type', 'unknown')
    r['no_nh']       = ev['no_nh']
    r['has_edgar']   = ev['has_edgar']
    r['burst_change_pct'] = ev.get('burst_change_pct', np.nan)
    results.append(r)

res = pd.DataFrame(results)
print(f"Trades simulados: {len(res)}")
print(f"  Grupo A (filtrado):  {res['filter_pass'].sum()}")
print(f"  Grupo B (excluidos): {(~res['filter_pass']).sum()}")


## 6. Resultados

In [ ]:
def run_scenario(sim_events, bars, label, setup_filter=None,
                 stop_pct=-0.08, trail_pct=None, target_pct=None,
                 max_entry_mso=MAX_ENTRY_MSO):
    """Corre una variante del simulador y retorna DataFrame de resultados."""
    evs = sim_events[sim_events['mso'] <= max_entry_mso].copy()
    if setup_filter:
        evs = evs[evs['setup_type'].isin(setup_filter)]
    results = []
    for _, ev in evs.iterrows():
        r = simulate_trade(
            ev['ticker'], ev['burst_date'], int(ev['mso']), bars,
            stop_pct=stop_pct, trail_pct=trail_pct, target_pct=target_pct,
        )
        if r is None: continue
        r['filter_pass'] = ev['filter_pass']
        r['setup_type']  = ev.get('setup_type', 'unknown')
        r['no_nh']       = ev['no_nh']
        r['has_edgar']   = ev['has_edgar']
        r['burst_change_pct'] = ev.get('burst_change_pct', np.nan)
        results.append(r)
    return pd.DataFrame(results)


def stats(df, label=''):
    if df.empty:
        return {'label': label, 'n': 0}
    wr   = (df['pnl_pct'] > 0).mean()
    avg  = df['pnl_pct'].mean()
    med  = df['pnl_pct'].median()
    wins = df.loc[df['pnl_pct'] > 0, 'pnl_pct'].mean() if (df['pnl_pct'] > 0).any() else 0
    loss = df.loc[df['pnl_pct'] < 0, 'pnl_pct'].mean() if (df['pnl_pct'] < 0).any() else 0
    pf   = abs(wins / loss) if loss else float('inf')
    stop_rate = df['exit_reason'].str.contains('STOP').mean()
    eod_rate  = (df['exit_reason'] == 'EOD').mean()
    return {
        'label': label, 'n': len(df),
        'wr': wr, 'avg_pnl': avg, 'median': med,
        'avg_win': wins, 'avg_loss': loss, 'pf': pf,
        'total_usd': df['pnl_usd'].sum(),
        'stop_rate': stop_rate, 'eod_rate': eod_rate,
        'mfe': df['mfe_pct'].mean(), 'mae': df['mae_pct'].mean(),
    }


def print_stats(s):
    if s['n'] == 0:
        print(f"  {s['label']}: sin datos"); return
    print(f"  {s['label']} (n={s['n']})")
    print(f"    WR={s['wr']:.1%}  AvgPnL={s['avg_pnl']:+.2f}%  Median={s['median']:+.2f}%  PF={s['pf']:.2f}x")
    print(f"    AvgWin={s['avg_win']:+.2f}%  AvgLoss={s['avg_loss']:+.2f}%  Total=${s['total_usd']:+,.0f}")
    print(f"    StopRate={s['stop_rate']:.1%}  EOD={s['eod_rate']:.1%}  MFE={s['mfe']:.2f}%  MAE={s['mae']:.2f}%")


# ── Escenarios ────────────────────────────────────────────────────────────────
print("=" * 65)
print("COMPARATIVA DE ESCENARIOS  (capital=$1,000/trade, MSO<=60)")
print("=" * 65)
print()

scenarios = [
    # (label, setup_filter, stop_pct, trail_pct)
    ("Baseline — todos, stop fijo -8%",          None,              -0.08, None),
    ("Todos — trailing -8%",                     None,              -0.15, 0.08),
    ("Todos — trailing -5%",                     None,              -0.10, 0.05),
    ("earnings_play — stop fijo -8%",            ['earnings_play'], -0.08, None),
    ("earnings_play — trailing -8%",             ['earnings_play'], -0.15, 0.08),
    ("earnings_play — trailing -5%",             ['earnings_play'], -0.10, 0.05),
    ("earnings_play + filtro — trailing -8%",    ['earnings_play'], -0.15, 0.08),  # filter_pass
]

all_stats = []
for label, setup_f, stop, trail in scenarios:
    df = run_scenario(sim_events, bars, label, setup_filter=setup_f, stop_pct=stop, trail_pct=trail)
    # Para el último escenario aplicar filtro adicional
    if 'filtro' in label:
        df = df[df['filter_pass']]
    s = stats(df, label)
    print_stats(s)
    print()
    all_stats.append((label, df, s))


## 7. Equity curve y distribución

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Burst T+0 Simulator  |  stop={STOP_PCT*100:.0f}%  entry_MSO<={MAX_ENTRY_MSO}', fontsize=13)

# ── Equity curves ─────────────────────────────────────────────────────────────
ax = axes[0, 0]
for label, mask, color in [
    ('Todos',         pd.Series([True]*len(res), index=res.index),  'steelblue'),
    ('Filtrado (A)',  res['filter_pass'],                            'green'),
    ('Excluidos (B)', ~res['filter_pass'],                           'tomato'),
]:
    grp = res[mask].sort_values('burst_date').reset_index(drop=True)
    if grp.empty: continue
    cum = grp['pnl_usd'].cumsum()
    ax.plot(cum.values, label=f"{label} (n={len(grp)})", color=color)
ax.set_title('Equity curve (P&L acumulado $)')
ax.set_xlabel('Trade #')
ax.set_ylabel('P&L $ acumulado')
ax.legend()
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# ── P&L distribution ─────────────────────────────────────────────────────────
ax = axes[0, 1]
bins = np.linspace(res['pnl_pct'].quantile(0.02), res['pnl_pct'].quantile(0.98), 40)
ax.hist(res.loc[~res['filter_pass'], 'pnl_pct'], bins=bins, alpha=0.5, color='tomato',  label='Excluidos')
ax.hist(res.loc[ res['filter_pass'], 'pnl_pct'], bins=bins, alpha=0.7, color='green',   label='Filtrados')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Distribución P&L % por trade')
ax.set_xlabel('P&L %')
ax.legend()

# ── Win rate por MSO de entrada ───────────────────────────────────────────────
ax = axes[1, 0]
res['mso_bucket'] = (res['entry_mso'] // 15) * 15  # buckets de 15 min
wr_mso = res.groupby('mso_bucket').apply(lambda x: pd.Series({
    'wr': (x['pnl_pct'] > 0).mean(),
    'n':  len(x),
    'avg': x['pnl_pct'].mean()
})).reset_index()
ax.bar(wr_mso['mso_bucket'], wr_mso['wr'], width=12, color='steelblue', alpha=0.8)
ax.axhline(0.5, color='red', linestyle='--', linewidth=0.8)
ax.set_title('Win rate por MSO de entrada (buckets 15 min)')
ax.set_xlabel('MSO (min desde apertura)')
ax.set_ylabel('Win rate')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))
for _, row in wr_mso.iterrows():
    ax.text(row['mso_bucket'], row['wr'] + 0.02, f"n={int(row['n'])}", ha='center', fontsize=7)

# ── MFE vs MAE scatter ────────────────────────────────────────────────────────
ax = axes[1, 1]
filt  = res[res['filter_pass']]
excl  = res[~res['filter_pass']]
ax.scatter(excl['mae_pct'], excl['mfe_pct'], alpha=0.4, color='tomato', s=20, label='Excluidos')
ax.scatter(filt['mae_pct'], filt['mfe_pct'], alpha=0.6, color='green',  s=25, label='Filtrados')
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
ax.axhline(abs(STOP_PCT)*100, color='black', linestyle=':', linewidth=0.8)
ax.set_title('MFE vs MAE por trade (%)')
ax.set_xlabel('MAE % (adverso, negativo = bajada)')
ax.set_ylabel('MFE % (favorable)')
ax.legend()

plt.tight_layout()
plt.savefig('figures/burst_t0_simulator.png', dpi=130, bbox_inches='tight')
plt.show()
print("Guardado en figures/burst_t0_simulator.png")


## 7. Ranking diario: ¿cuál ticker elegir cuando hay varios?

In [ ]:
# Cuando hay múltiples tickers en Top Gainers el mismo día,
# simular una regla de selección: tomar solo 1 ticker por día
# según prioridad: earnings_play > insider_driven > other_edgar > accumulation_long
# y dentro del setup, el que apareció primero (MSO menor)

SETUP_PRIORITY = {
    'earnings_play':    1,
    'insider_driven':   2,
    'other_edgar':      3,
    'accumulation_long':4,
    'clean_momentum':   5,
    'single_catalyst':  6,
    'unknown':          9,
}

# Simulable events con trailing stop -8%
trail_all = run_scenario(sim_events, bars, 'all', stop_pct=-0.15, trail_pct=0.08)
trail_all['setup_priority'] = trail_all['setup_type'].map(SETUP_PRIORITY).fillna(9)

# Merge con entry_mso desde sim_events
entry_mso_map = sim_events.set_index(['ticker','burst_date'])['mso'].to_dict()
trail_all['first_mso'] = trail_all.apply(
    lambda r: entry_mso_map.get((r['ticker'], r['burst_date']), 999), axis=1
)

# Por día: elegir el mejor ticker según prioridad setup → MSO más temprano
best_per_day = (
    trail_all
    .sort_values(['burst_date', 'setup_priority', 'first_mso'])
    .groupby('burst_date')
    .first()
    .reset_index()
)

print("=" * 65)
print("SELECCIÓN: MEJOR TICKER POR DÍA (trailing -8%)")
print("=" * 65)
print(f"Días con al menos 1 ticker simulable: {trail_all['burst_date'].nunique()}")
print(f"Selección 1 ticker/día: {len(best_per_day)} trades")
print()
s_best = stats(best_per_day, "1 ticker/día — mejor setup + MSO")
print_stats(s_best)
print()

# Comparar con selección aleatoria (avg de todos)
s_all = stats(trail_all, "Todos los tickers (trailing -8%)")
print_stats(s_all)
print()

# Distribución de setup_type en la selección
print("Setup type elegido:")
print(best_per_day['setup_type'].value_counts().to_string())
print()

# Equity curve comparativa
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
cum_best = best_per_day.sort_values('burst_date')['pnl_usd'].cumsum()
cum_all  = trail_all.sort_values('burst_date').groupby('burst_date')['pnl_usd'].mean().cumsum()
ax.plot(cum_best.values, color='green',    label=f"1 ticker/día — mejor setup (n={len(best_per_day)})")
ax.plot(cum_all.values,  color='steelblue',label=f"Promedio todos los días (n días={len(cum_all)})", alpha=0.7)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Equity curve — selección vs todos (trailing -8%)')
ax.set_xlabel('Día de trading')
ax.set_ylabel('P&L $ acumulado')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))

# P&L por setup en la selección
ax = axes[1]
st_pnl = best_per_day.groupby('setup_type')['pnl_pct'].agg(['mean','median','count'])
st_pnl = st_pnl.sort_values('mean', ascending=True)
colors = ['green' if v > 0 else 'tomato' for v in st_pnl['mean']]
ax.barh(st_pnl.index, st_pnl['mean'], color=colors, alpha=0.8)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Avg P&L % por setup (selección diaria)')
ax.set_xlabel('Avg P&L %')
for i, (idx, row) in enumerate(st_pnl.iterrows()):
    ax.text(row['mean'] + 0.2, i, f"n={int(row['count'])}", va='center', fontsize=9)

plt.tight_layout()
plt.savefig('figures/burst_t0_ranking.png', dpi=130, bbox_inches='tight')
plt.show()
print("Guardado en figures/burst_t0_ranking.png")


## 8. Análisis de sensibilidad (stop %)

In [ ]:
# Barrer stop loss de -3% a -15% y ver cómo cambian WR y PF
stop_sweep = np.arange(-0.03, -0.16, -0.01)
rows_sweep = []
for stop in stop_sweep:
    for group_name, mask in [('A_filtrado', res['filter_pass']), ('B_todos', pd.Series(True, index=res.index))]:
        grp = res[mask]
        r_list = []
        for _, ev in sim_events[sim_events['filter_pass'] == (group_name == 'A_filtrado')].iterrows():
            r = simulate_trade(ev['ticker'], ev['burst_date'], int(ev['mso']), bars, stop_pct=stop)
            if r: r_list.append(r)
        if not r_list: continue
        df_s = pd.DataFrame(r_list)
        wr = (df_s['pnl_pct'] > 0).mean()
        wins   = df_s.loc[df_s['pnl_pct'] > 0, 'pnl_pct'].mean() if (df_s['pnl_pct'] > 0).any() else 0
        losses = df_s.loc[df_s['pnl_pct'] < 0, 'pnl_pct'].mean() if (df_s['pnl_pct'] < 0).any() else 0
        pf = abs(wins / losses) if losses else float('inf')
        rows_sweep.append({'stop_pct': round(stop*100,0), 'group': group_name, 'wr': wr, 'pf': pf, 'avg_pnl': df_s['pnl_pct'].mean()})

sweep = pd.DataFrame(rows_sweep)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (col, label) in enumerate([('wr','Win Rate'), ('pf','Profit Factor'), ('avg_pnl','Avg P&L %')]):
    ax = axes[i]
    for grp_name, color in [('A_filtrado','green'), ('B_todos','steelblue')]:
        d = sweep[sweep['group'] == grp_name]
        if d.empty: continue
        ax.plot(d['stop_pct'], d[col], marker='o', label=grp_name, color=color)
    ax.set_title(label)
    ax.set_xlabel('Stop %')
    ax.legend()
    ax.axhline(0.5 if col=='wr' else (1.0 if col=='pf' else 0), color='red', linestyle='--', linewidth=0.8)

plt.suptitle('Sensibilidad al stop loss — filtrado vs todos', fontsize=12)
plt.tight_layout()
plt.savefig('figures/burst_t0_sensitivity.png', dpi=130, bbox_inches='tight')
plt.show()

# Tabla resumen
print(sweep.pivot_table(index='stop_pct', columns='group', values=['wr','pf','avg_pnl']).round(3).to_string())
